# Sensitivity to physical model coefficients

Differentiate a fixed-noise harvesting trajectory with respect to selected physical coefficients. Keep the integration grid and random key fixed. Differentiating the whole configuration would also trace discrete grid settings, which require concrete values.

The shared `make_trajectory` method validates arrays eagerly. This example differentiates the low-level `simulate` kernel. It computes trajectory sensitivities, not derivatives of an estimated event time.

In [ ]:
from dataclasses import replace

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from tipbox.models import MayHarvesting, MayHarvestingConfig

baseline = MayHarvestingConfig()
parameter_names = ("r", "k", "s", "h_max", "sigma")
parameters = jnp.asarray(
    [
        getattr(
            baseline,
            name,
        )
        for name in parameter_names
    ]
)


def population_for(coefficients):
    config = replace(
        baseline,
        **dict(
            zip(
                parameter_names,
                coefficients,
            )
        ),
    )
    model = MayHarvesting(config)
    _, trajectory, _, _ = model.simulate(
        key=jax.random.PRNGKey(42),
        total_len=400,
        sigma_tilde=1.0,
        null_simulation=False,
        initial_state=jnp.asarray(0.8),
    )
    return trajectory[:, 0]


## Compute a trajectory Jacobian

A fixed key gives the same Gaussian innovations for each coefficient perturbation. Keeping `dt` and the output grid outside the differentiated vector avoids integer/clock tracing. An interval count of 400 returns 401 samples, including the initial state.

In [ ]:
population = population_for(parameters)
# Forward mode suits five inputs and hundreds of trajectory outputs.
sensitivity = jax.jacfwd(population_for)(parameters)
print(population.shape)  # (401,)
print(sensitivity.shape)  # (401, 5)


## Plot state and sensitivity

Each derivative has biomass divided by its coefficient's units. Their raw magnitudes are not a unitless importance ranking. The nonnegative population projection is retained, so gradients can change at that numerical boundary.

In [ ]:
time = baseline.t_start + jnp.arange(population.size)
fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4),
)
axes[0].plot(
    time,
    population,
    color="#216a91",
)
axes[0].set(
    xlabel="Time (model units)",
    ylabel="Population",
    title="Fixed-noise trajectory",
)
for index, name in enumerate(parameter_names):
    axes[1].plot(
        time,
        sensitivity[:, index],
        label=name,
    )
axes[1].set(
    xlabel="Time (model units)",
    ylabel="∂population / ∂coefficient",
    title="Physical-parameter sensitivities",
)
axes[1].legend()
fig.tight_layout()
plt.show()


See [model controls](../docs/usage.rst) and the [harvesting equation/defaults](../docs/models/harvesting.rst). For event-aligned labeled datasets and ROC evaluation, use the [batch notebook](01_generate_batch.ipynb).